Website Brochure

In [39]:
import os
import requests
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from bs4 import BeautifulSoup

In [40]:
MODEL = 'llama3.2'
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

In [41]:
# Scrapper
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

# get website content
def fetch_website_contents(url):
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]

# Get links within website
def fetch_website_links(url):
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    links = [link.get("href") for link in soup.find_all("a")]
    return [link for link in links if link]


In [42]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as lainks to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [43]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [44]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [45]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [46]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [47]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:2_000] # Truncate if more than 5,000 characters
    return user_prompt

In [48]:
def create_brochure(company_name, url):
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [49]:
def stream_brochure(company_name, url):
    stream = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [50]:
stream_brochure("Ollama", "https://ollama.com/")

Selecting relevant links for https://ollama.com/ by calling llama3.2
Found 5 relevant links


**Ollama Brochure**

Welcome to Ollama, a pioneering technology company that's revolutionizing the way we interact with data. Our mission is to empower users to automate their work and unlock new possibilities.

**About Us**

Ollama is built on the concept of "Open Models" - a cutting-edge approach to artificial intelligence that combines power with accessibility. Our cloud platform provides access to faster, larger models when you need them, allowing users to scale up or down as needed.

**Our Values**

At Ollama, we're passionate about democratizing AI and making it more accessible to everyone. We believe in community-driven innovation, collaborative growth, and a culture of continuous learning. Our values are centered around empowering users to automate their work, unlocking new possibilities, and pushing the boundaries of what's possible.

**Products and Services**

We offer a range of products and services designed to help you achieve your goals:

* **OpenClaw**: The easiest way to build with open models, allowing you to automate your work in minutes.
* **Cloud Models**: Access faster, larger models when you need them, on datacenter-grade hardware.
* **Pro and Max Plans**: Scale up or down to suit your needs, with flexible pricing options.

**Customer Success**

We're proud to serve a range of customers, from businesses looking to automate their workflow to individuals seeking to unlock new possibilities. Our cloud platform is designed to be flexible, scalable, and secure, ensuring the integrity of your data.

**Careers and Opportunities**

Join our team of innovators and problem-solvers! We're always looking for talented individuals who share our passion for AI and collaboration. Visit our Careers page to learn more about our job openings and company culture.

**Get Started Today**

Take the first step towards automating your work with Ollama. Download our software, sign up for a free trial, or explore our resources to learn more about OpenClaw and our cloud platform.

**Stay Connected**

Follow us on Twitter and join our community of innovators!

[Your Twitter handle]

**Copyright 2026 Ollama Inc.**

In [51]:
stream_brochure("Anthropic", "https://www.anthropic.com/")

Selecting relevant links for https://www.anthropic.com/ by calling llama3.2
Found 20 relevant links


# Anthropic: Securing the Benefits of AI

Anthropic is a public benefit corporation dedicated to ensuring that artificial intelligence (AI) is developed and used in ways that prioritize safety, security, and humanity. Our mission is driven by the understanding that AI will have a profound impact on the world, and it's our responsibility to mitigate its risks while maximizing its benefits.

## About Us

At Anthropic, we believe in fostering a culture of transparency, accountability, and collaboration. We're committed to creating an inclusive and diverse community of researchers, engineers, and users who share our vision for a safe and equitable AI future.

### Our Commitments

* To prioritize the well-being and dignity of individuals and communities affected by AI
* To ensure that AI systems are designed, developed, and deployed in ways that reduce harm and promote benefits
* To promote diversity, equity, and inclusion in the field of AI research and practice

## Our Work

Anthropic is working on several projects to secure the benefits of AI for all. Some of our notable initiatives include:

* **Project Glasswing**: We're developing secure critical software for the AI era, focusing on protecting against emerging threats and vulnerabilities.
* **Claude**: Our flagship AI platform provides a powerful tool for coding, agents, and professional work. Claude Opus 4.6 is one of the world's most advanced models, enabling developers to create safer, more effective AI systems.

## Our Team

Our team of experts comes from diverse backgrounds in AI research, engineering, and policy-making. We're passionate about creating a better future for all through our collective work.

### Careers

If you share our vision and values, we invite you to join our team! Check out our career page for available positions and learn more about what it's like to work with us.

## Stay Informed

Stay up-to-date on the latest news and developments from Anthropic. Follow us on social media or visit our blog for insights into AI research, policy, and best practices.

### Contact Us

Get in touch with us to learn more about our mission, products, or careers opportunities.

# Join the conversation
Twitter: [insert link]
Facebook: [insert link]
Newsletter: [insert link]

Note: This brochure is based on the information available on the company's website, including its landing page, product pages, and other relevant content.